### 1. Access elements from the JSON object
### 2.Deduplicate Array Elements
### 3.Explode Arrays
### 4. Write the Transformed Data to Silver Schema

In [0]:
df_order_JsonObject = spark.read.table("gizmobox_nara.silver.py_orders_json")

In [0]:
%sql
Select * from gizmobox_nara.silver.orders_json

### !. Access elements from JSON Objects

In [0]:
df_accessing_elements_in_json = (
    df_order_JsonObject
    .select("json_value.order_id",
             "json_value.order_status",
             "json_value.payment_method",
              "json_value.total_amount",
               "json_value.transaction_timestamp",
               "json_value.items"
               )
)
display(df_accessing_elements_in_json)

In [0]:
%sql
SELECT json_value.order_id,
       json_value.order_status,
       json_value.payment_method,
       json_value.total_amount,
       json_value.transaction_timestamp,
       json_value.items
       FROM gizmobox_nara.silver.orders_json
       

### 2.deduplicate Array Elements - spark function(array_distinct)

In [0]:
df_duplicates_order_json = (
    df_accessing_elements_in_json
    .select(
        "json_value.order_id",
        "json_value.order_status",
        "json_value.payment_method",
        "json_value.total_amount",
        "json_value.transaction_timestamp",
        array_distinct("json_value.items") AS items
    )
)
display(df_duplicates_order_json)

In [0]:
%sql
SELECT json_value.order_id,
       json_value.order_status,
       json_value.payment_method,
       json_value.total_amount,
       json_value.transaction_timestamp,
       array_distinct(json_value.items) AS items
       FROM gizmobox_nara.silver.orders_json

### Explode Array-function(explode())

In [0]:
df_explode_item = (
    df_distinct_order_json
    .select(
        "json_value.order_id",
        "json_value.order_status",
        "json_value.payment_method",
        "json_value.total_amount",
        "json_value.transaction_timestamp",
        "json_value.customer_id",
        explode(array_distinct("json_value.items")) AS item
    )
)
display(df_explode_item)

In [0]:
%sql
DROP VIEW IF EXISTS tv_orders_exploded;
CREATE OR REPLACE TEMP VIEW tv_orders_exploded AS
SELECT json_value.order_id,
       json_value.order_status,
       json_value.payment_method,
       json_value.total_amount,
       json_value.transaction_timestamp,
       json_value.customer_id,
       explode(array_distinct(json_value.items)) AS item
       FROM gizmobox_nara.silver.orders_json

In [0]:
%sql
SELECT * FROM tv_orders_exploded

In [0]:
df_individual_items = (
    df_explode_item
    .select(
        "json_value.order_id",
        "json_value.order_status",
        "json_value.payment_method",
        "json_value.total_amount",
        "json_value.transaction_timestamp",
        "json_value.customer_id",
        "item.category",
        "item.details.brand",
        "item.details.color",
        "item.item_id",
        "item.name",
        "item.price",
        "item.quantity",
    )
)
display(df_individual_items)

In [0]:
%sql
SELECT order_id,
       order_status,
       payment_method,
       total_amount,
       transaction_timestamp,
       customer_id,
       item.category,
       item.details.brand,
       item.details.color,
       item.item_id,
       item.name,
       item.price,
       item.quantity
       FROM tv_orders_exploded

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gizmobox_nara.silver.orders
 AS
 SELECT order_id,
       order_status,
       payment_method,
       total_amount,
       transaction_timestamp,
       customer_id,
       item.category,
       item.details.brand,
       item.details.color,
       item.item_id,
       item.name,
       item.price,
       item.quantity
       FROM tv_orders_exploded


In [0]:
%sql
SELECT * FROM gizmobox_nara.silver.orders